# Strategic Investment Banking OS — Transparent Manual Loop 002

This notebook performs the second controlled experiment openly:

1. locate and snapshot the independent Obsidian vault;
2. inspect the current universe;
3. normalize or verify fictional company `SYN-102`;
4. declare a synthetic AI-infrastructure investment boom;
5. show every scenario-score component;
6. identify the affected company subgraph;
7. produce candidate growth-capital, infrastructure-financing and strategic-M&A lenses;
8. compare the result with Manual Loop 001;
9. preview every proposed file mutation;
10. write only after explicit human approval.

> All companies, assumptions and results are synthetic. This is an architecture and governance test—not investment advice, a market forecast or a live mandate recommendation.

The notebook uses only Python's standard library. It makes no LLM call, requires no API key and contacts nobody.


## Why this is a real second test

Loop 001 asked a defensive credit question. Loop 002 asks an expansion question. A useful operating system should change both the affected subgraph and the banking responses when the causal scenario changes, while preserving the first run as historical evidence.


In [ ]:
# 1. Imports and explicit safety configuration

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from html import escape
import csv, hashlib, io, json, math, os, shutil, tempfile

from IPython.display import display, HTML, Markdown

VAULT_NAME = "Alejandro-Reynoso-Investment-Banking-Vault"
LOOP_ID = "LOOP-002-COLAB"
RUN_DATE = "2026-07-17"

# Keep False for a complete dry run. Change only after reviewing every output.
COMMIT_CHANGES = False
BACKUP_BEFORE_WRITE = True

print("Loop:", LOOP_ID)
print("COMMIT_CHANGES:", COMMIT_CHANGES)


In [ ]:
# 2. Mount Google Drive in Colab and resolve this separate vault

import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY_DRIVE = Path("/content/drive/MyDrive")
else:
    MY_DRIVE = Path.cwd()

VAULT = MY_DRIVE / VAULT_NAME
if not VAULT.exists():
    candidates = list(MY_DRIVE.glob(f"**/{VAULT_NAME}"))
    if len(candidates) != 1:
        raise FileNotFoundError(f"Expected one {VAULT_NAME}; found {candidates}")
    VAULT = candidates[0]

print("Vault:", VAULT)
print("Home present:", (VAULT / "00 Home.md").exists())


In [ ]:
# 3. Snapshot files and load company_master.csv without pandas

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

files_before = sorted(p for p in VAULT.rglob("*") if p.is_file())
manifest_before = {str(p.relative_to(VAULT)): {"bytes": p.stat().st_size, "sha256": sha256(p)} for p in files_before}

MASTER_CSV = VAULT / "Data" / "company_master.csv"
NUMERIC_FIELDS = {
    "founded", "employees", "revenue_prev_usd_m", "revenue_usd_m", "revenue_growth_pct",
    "ebitda_usd_m", "ebitda_margin_pct", "net_income_usd_m", "free_cash_flow_usd_m",
    "cash_usd_m", "debt_usd_m", "net_debt_usd_m", "enterprise_value_usd_m",
    "equity_value_usd_m", "ev_revenue", "ev_ebitda", "pe_ratio", "net_leverage",
    "roic_pct", "recurring_revenue_pct", "top_customer_concentration_pct", "moat_score",
    "management_score", "execution_risk_score", "esg_score",
}

def parse_number(value):
    return None if value is None or value == "" or str(value).lower() == "none" else float(value)

with MASTER_CSV.open(encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    fieldnames = list(reader.fieldnames or [])
    records = []
    for raw in reader:
        row = dict(raw)
        for field in NUMERIC_FIELDS:
            if field in row:
                row[field] = parse_number(row[field])
        records.append(row)

print("Files before run:", len(files_before))
print("Company rows loaded:", len(records))
print("Unique company IDs:", len({r['id'] for r in records}))


In [ ]:
# 4. Reusable transparent table renderer and universe inspection

def display_table(rows, columns, title=None):
    if title:
        display(Markdown(f"### {title}"))
    head = "".join(f"<th style='padding:6px'>{escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td style='padding:6px;border-top:1px solid #ddd'>{escape(str(row.get(c, '')))}</td>" for c in columns) + "</tr>" for row in rows)
    display(HTML(f"<div style='overflow-x:auto'><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>"))

display_table([{"Sector": k, "Companies": v} for k, v in sorted(Counter(r["sector"] for r in records).items())], ["Sector", "Companies"], "Universe before intake")


## Idempotent intake rule

- If `SYN-102` is absent, propose adding it.
- If it is already `VoltEdge Thermal Systems`, verify it and do not duplicate it.
- If that ID belongs to anything else, stop.

This makes repeated dry runs safe.


In [ ]:
# 5. Define and validate fictional company SYN-102

SYN102 = {
    "id": "SYN-102", "name": "VoltEdge Thermal Systems", "sector": "Industrials",
    "subsector": "Data center thermal management", "theme": "Digital Infrastructure",
    "region": "North America", "stage": "Scale-up", "investment_style": "Growth",
    "ownership": "Growth-equity backed", "founded": 2020, "employees": 640,
    "business_model": "Equipment and service contracts", "market_position": "Emerging challenger",
    "description": "Fictional high-density liquid-cooling and thermal-management systems for AI data centers.",
    "revenue_prev_usd_m": 105.0, "revenue_usd_m": 147.0, "revenue_growth_pct": 40.0,
    "ebitda_usd_m": 14.7, "ebitda_margin_pct": 10.0, "net_income_usd_m": 4.2,
    "free_cash_flow_usd_m": -6.5, "cash_usd_m": 35.0, "debt_usd_m": 42.0,
    "net_debt_usd_m": 7.0, "enterprise_value_usd_m": 420.0, "equity_value_usd_m": 413.0,
    "ev_revenue": 2.86, "ev_ebitda": 28.57, "pe_ratio": 98.33, "net_leverage": 0.48,
    "roic_pct": 8.5, "recurring_revenue_pct": 45, "top_customer_concentration_pct": 38,
    "moat_score": 8, "management_score": 7, "execution_risk_score": 7, "esg_score": 78,
    "profitability": "Profitable", "transaction_type": "Growth capital",
    "transaction_rationale": "Fund capacity, working capital and service-network expansion.",
}

same_id = [r for r in records if r["id"] == "SYN-102"]
if same_id and same_id[0]["name"] != SYN102["name"]:
    raise ValueError("STOP: SYN-102 belongs to another company.")
intake_status = "ALREADY PRESENT — verify only" if same_id else "PROPOSED CREATE"
analysis_records = records if same_id else records + [SYN102]

ev_check = SYN102["equity_value_usd_m"] + SYN102["debt_usd_m"] - SYN102["cash_usd_m"]
checks = [
    {"Check": "EV = Equity + Debt - Cash", "Pass": abs(ev_check - SYN102["enterprise_value_usd_m"]) < 1e-9},
    {"Check": "Unique IDs", "Pass": len({r['id'] for r in analysis_records}) == len(analysis_records)},
    {"Check": "Unique names", "Pass": len({r['name'] for r in analysis_records}) == len(analysis_records)},
    {"Check": "SYN-102 appears once", "Pass": sum(r['id'] == 'SYN-102' for r in analysis_records) == 1},
]
display_table(checks, ["Check", "Pass"], f"Intake validation — {intake_status}")
assert all(c["Pass"] for c in checks)


## Declared synthetic scenario

- Enterprise AI adoption accelerates.
- Data-center capital expenditure rises 30%.
- Cooling, power, grid, storage and fiber become constraints.
- Capital markets remain selectively open for capacity expansion.

The score below measures only relevance to this scenario. It is not a valuation or investment-quality score.


In [ ]:
# 6. Expose every scenario-score component

SECTOR_WEIGHTS = {"Technology": 15, "Real Estate": 18, "Energy and Utilities": 18, "Telecom and Media": 15, "Industrials": 12, "Mobility and Logistics": 5}
KEYWORDS = ("data center", "cloud", "artificial intelligence", "ai ", "grid", "fiber", "battery", "cooling", "power", "inference", "thermal", "telecom tower")

def score_breakdown(c):
    text = f"{c['subsector']} {c['description']}".lower()
    theme = 25 if c["theme"] == "Digital Infrastructure" else 10 if c["theme"] in {"Supply Chain Resilience", "Decarbonization"} else 0
    sector = SECTOR_WEIGHTS.get(c["sector"], 0)
    keyword = 20 if any(k in text for k in KEYWORDS) else 0
    growth = min(15, max(0, c["revenue_growth_pct"]) * 0.3)
    recurring = min(10, c["recurring_revenue_pct"] / 10)
    moat = c["moat_score"] * 1.2
    capital_need = 8 if c["free_cash_flow_usd_m"] < 0 else 0
    execution_penalty = max(0, c["execution_risk_score"] - 5) * 3
    raw = 10 + theme + sector + keyword + growth + recurring + moat + capital_need - execution_penalty
    return {"id": c["id"], "company": c["name"], "sector": c["sector"], "base": 10,
            "theme_component": theme, "sector_component": sector, "keyword_component": keyword,
            "growth_component": round(growth, 2), "recurring_component": round(recurring, 2),
            "moat_component": round(moat, 2), "capital_need_component": capital_need,
            "execution_penalty": execution_penalty, "raw_total": round(raw, 2),
            "final_score": min(95, round(raw))}

breakdowns = [score_breakdown(r) for r in analysis_records]
ranked = sorted(breakdowns, key=lambda b: (b["final_score"], b["raw_total"]), reverse=True)
affected = ranked[:8]
display_table(ranked[:15], ["company", "sector", "base", "theme_component", "sector_component", "keyword_component", "growth_component", "recurring_component", "moat_component", "capital_need_component", "execution_penalty", "raw_total", "final_score"], "Complete score decomposition")


In [ ]:
# 7. Translate the top subgraph into candidate banking lenses

by_name = {r["name"]: r for r in analysis_records}
def mandate_for(c):
    if c["name"] == SYN102["name"] or (c["sector"] == "Technology" and c["stage"] in {"Startup", "Scale-up"}):
        return "Growth capital"
    if c["sector"] in {"Real Estate", "Energy and Utilities"}:
        return "Infrastructure financing"
    return "Strategic M&A"

opportunities = []
# Preserve the raw top six, then add two disclosed coverage-diversity candidates
# so infrastructure finance is tested without altering any relevance score.
top_energy = next(r for r in ranked if r["sector"] == "Energy and Utilities")
data_center_operator = next(r for r in ranked if r["company"] == "Distrito Data Centers")
opportunity_results = affected[:6] + [top_energy, data_center_operator]
for number, result in enumerate(opportunity_results, start=7):
    c = by_name[result["company"]]
    opportunities.append({"opportunity_id": f"OPP-{number:03d}", "company": c["name"],
                          "mandate": mandate_for(c), "score": result["final_score"],
                          "why": f"Theme {result['theme_component']}; sector {result['sector_component']}; keyword {result['keyword_component']}; growth {result['growth_component']}; capital need {result['capital_need_component']}; risk penalty {result['execution_penalty']}."})
display_table(opportunities, ["opportunity_id", "company", "mandate", "score", "why"], "Candidate opportunity lenses")


In [ ]:
# 8. Compare with the frozen Loop 001 affected subgraph

loop1 = {"DeltaMed Systems", "Estela Robotics", "Croma Studios", "Dynamo Controls", "Galen Materials", "Amapola Living", "Ember Storage", "Huerto Labs"}
loop2 = {r["company"] for r in affected}
overlap = sorted(loop1 & loop2)
jaccard = len(loop1 & loop2) / len(loop1 | loop2)
comparison = [
    {"Measure": "Top-eight overlap", "Result": len(overlap)},
    {"Measure": "Jaccard similarity", "Result": round(jaccard, 3)},
    {"Measure": "Loop 001 dominant lens", "Result": "Restructuring / liability management"},
    {"Measure": "Loop 002 mandate mix", "Result": dict(Counter(o['mandate'] for o in opportunities))},
]
display_table(comparison, ["Measure", "Result"], "Generalization test")
print("Overlapping companies:", overlap or "None")


In [ ]:
# 9. Build auditable companion artifacts in memory — still no writes

def csv_text(rows, columns):
    buf = io.StringIO(); writer = csv.DictWriter(buf, fieldnames=columns, extrasaction="ignore")
    writer.writeheader(); writer.writerows(rows); return buf.getvalue()

score_columns = list(breakdowns[0].keys())
affected_lines = "\n".join(f"- {r['company']}: {r['final_score']}/100" for r in affected)
pipeline_rows = "\n".join(f"| {o['opportunity_id']} | {o['company']} | {o['mandate']} | {o['score']}/100 | Candidate |" for o in opportunities)

scenario_md = f'''---
type: scenario-reproduction
scenario_id: SCN-002-COLAB
synthetic: true
---
# SCN-002 — Colab Transparent Reproduction

Assumptions: AI adoption accelerates; data-center capex rises 30%; cooling, power, grid, storage and fiber constrain expansion.

## Affected subgraph
{affected_lines}
'''
pipeline_md = f'''---
type: opportunity-pipeline-reproduction
loop_id: {LOOP_ID}
synthetic: true
---
# Opportunity Pipeline — Loop 002 Colab Reproduction

| ID | Company | Mandate | Score | Status |
|---|---|---|---:|---|
{pipeline_rows}
'''
comparison_md = f'''---
type: generalization-review-reproduction
loop_id: {LOOP_ID}
---
# Loop 001 vs Loop 002 — Colab Reproduction

- Top-eight overlap: {len(overlap)}
- Jaccard similarity: {jaccard:.3f}
- Overlapping companies: {', '.join(overlap) if overlap else 'None'}
- Loop 002 mandate mix: {dict(Counter(o['mandate'] for o in opportunities))}
'''
audit_md = f'''---
type: audit-record
audit_id: AUD-002N
loop_id: {LOOP_ID}
---
# AUD-002N — Colab Transparent Reproduction

- Records analyzed: {len(analysis_records)}
- SYN-102: {intake_status}
- Affected companies: {len(affected)}
- Candidate opportunities: {len(opportunities)}
- External sources: none
- LLM calls: none
- External actions: none
- Human approval required: yes
'''
relationship_md = '''---
type: relationship-map-reproduction
company: VoltEdge Thermal Systems
synthetic: true
---
# VoltEdge Thermal Systems — Colab Relationship Map

- [[Companies/Distrito Data Centers|Distrito Data Centers]] — customer hypothesis.
- [[Companies/TensorPeak|TensorPeak]] — AI-compute demand driver.
- [[Companies/Ember Storage|Ember Storage]] — power-resilience complement.
- [[Companies/DawnGrid|DawnGrid]] — grid complement.
- [[Companies/Beacon Fiber|Beacon Fiber]] — connectivity partner hypothesis.
'''

planned_writes = {
    Path("Data/loop_002_colab_scoring_breakdown.csv"): csv_text(breakdowns, score_columns),
    Path("Scenarios/SCN-002 - Colab Transparent Reproduction.md"): scenario_md,
    Path("Opportunities/Opportunity Pipeline - Loop 002 Colab Reproduction.md"): pipeline_md,
    Path("Audit/AUD-002N - Colab Transparent Reproduction.md"): audit_md,
    Path("Audit/Loop 001 vs Loop 002 - Colab Reproduction.md"): comparison_md,
    Path("Relationships/VoltEdge Thermal Systems - Colab Reproduction.md"): relationship_md,
}

for o in opportunities:
    safe = o["company"].replace("/", "-")
    card = f'''---
type: opportunity-reproduction
opportunity_id: {o['opportunity_id']}
company: "{o['company']}"
mandate: "{o['mandate']}"
score: {o['score']}
human_approval: required
synthetic: true
---
# {o['opportunity_id']} — {o['company']}

**Mandate lens:** {o['mandate']}

{o['why']}

Synthetic candidate; human diligence and approval required.
'''
    planned_writes[Path(f"Opportunities/{o['opportunity_id']} - {safe} - Loop 002 Colab Reproduction.md")] = card

print("Artifacts held in memory:", len(planned_writes))
print("Files written: 0")


In [ ]:
# 10. Preview the exact mutation manifest and hashes

mutation_manifest = []
for relative, content in planned_writes.items():
    target = VAULT / relative
    new_bytes = content.encode("utf-8")
    new_hash = hashlib.sha256(new_bytes).hexdigest()
    action = "CREATE" if not target.exists() else ("UNCHANGED" if sha256(target) == new_hash else "UPDATE")
    mutation_manifest.append({"action": action, "path": str(relative), "new_bytes": len(new_bytes), "new_sha256": new_hash})
display_table(mutation_manifest, ["action", "path", "new_bytes", "new_sha256"], "Proposed file mutations")
print("COMMIT_CHANGES:", COMMIT_CHANGES)


## Human commit gate

Run the entire notebook once with `COMMIT_CHANGES = False`. Review the intake checks, every scoring component, the affected subgraph, mandate mix, loop comparison and mutation manifest. Only then may you change the configuration to `True`. Existing targets are backed up first.


In [ ]:
# 11. Commit atomically only after explicit authorization

def atomic_write(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=path.name + ".", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f: f.write(content.rstrip() + "\n")
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp): os.unlink(tmp)

committed = []
if not COMMIT_CHANGES:
    print("DRY RUN COMPLETE — no files were written.")
else:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    backup_root = VAULT / "Audit" / "Backups" / f"{LOOP_ID}-{timestamp}"
    for relative, content in planned_writes.items():
        target = VAULT / relative
        if target.exists() and sha256(target) == hashlib.sha256(content.encode()).hexdigest():
            committed.append({"action": "UNCHANGED", "path": str(relative)}); continue
        if target.exists() and BACKUP_BEFORE_WRITE:
            backup = backup_root / relative; backup.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(target, backup)
        action = "UPDATE" if target.exists() else "CREATE"
        atomic_write(target, content); committed.append({"action": action, "path": str(relative)})

    if not same_id:
        if BACKUP_BEFORE_WRITE:
            backup = backup_root / "Data/company_master.csv"; backup.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(MASTER_CSV, backup)
        with MASTER_CSV.open("a", encoding="utf-8", newline="") as f:
            csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore").writerow(SYN102)
        committed.append({"action": "APPEND", "path": "Data/company_master.csv"})
    display_table(committed, ["action", "path"], "Committed changes")
    print("Backup root:", backup_root)


In [ ]:
# 12. Final validation and next-session hot-cache payload

validation = {
    "analysis_company_count": len(analysis_records),
    "unique_ids": len({r['id'] for r in analysis_records}) == len(analysis_records),
    "syn102_once": sum(r['id'] == 'SYN-102' for r in analysis_records) == 1,
    "affected_subgraph_size": len(affected),
    "candidate_opportunities": len(opportunities),
    "score_components_visible": all("raw_total" in r and "execution_penalty" in r for r in breakdowns),
    "writes_authorized": COMMIT_CHANGES,
}
display_table([{"Validation": k, "Result": v} for k, v in validation.items()], ["Validation", "Result"], "Final validation")
assert validation["unique_ids"] and validation["syn102_once"]

hot_cache_payload = {
    "loop_id": LOOP_ID,
    "company_universe": len(analysis_records),
    "new_company": "SYN-102 — VoltEdge Thermal Systems",
    "scenario": "Synthetic AI-infrastructure investment boom",
    "affected_companies": [r["company"] for r in affected],
    "candidate_opportunities": [f"{o['opportunity_id']} — {o['company']} — {o['mandate']}" for o in opportunities],
    "loop1_loop2_jaccard": round(jaccard, 3),
    "external_actions": 0,
    "human_approval_required": True,
    "next_control": "Add source provenance before real data or automation.",
}
print(json.dumps(hot_cache_payload, indent=2, ensure_ascii=False))


## What this notebook proves—and what it does not

It makes the second loop reproducible: the intake, scenario, scoring formula, affected graph, mandate translation, comparison, mutations and approval gate are all visible.

It does not prove that the score is economically calibrated, that any company is attractive, or that an actual mandate exists. The next baby step is evidence provenance: every real claim should point to a dated source, carry confidence, and remain separable from analyst inference.
